# Phase 2a — Custom Scoring

nflverse ships `fantasy_points` (standard) and `fantasy_points_ppr` (full PPR).
This league is **0.5 PPR with several non-standard rules**, so neither column is
a valid model target or a valid baseline against Sleeper's projections.

This notebook builds and validates a scorer that reproduces the league's rules
exactly, by diffing against what Sleeper actually awarded in completed weeks.

**Validated result:** 100% exact match on 2025 weeks 5, 8, 10, 12, 15 — every
rostered player, every position.

**Rules discovered by validation** (none of them documented anywhere):

| Rule | What it actually does |
|---|---|
| `fum` | Counts `fumbles_total`, not the sum of rushing/receiving/sack fumbles |
| `fum` + `fum_lost` | Stack — a lost fumble costs −2, a self-recovered one −1 |
| `fum_rec` | Does **not** apply to offensive players (19/19 confirmed) |
| `fgm_yds_over_30` | Per kick, not on aggregate distance |
| `xpmiss` | Blocked PATs count as misses |
| `fgmiss` | Only applies to misses under 50 yards |
| `pass_int_td` | Needs play-by-play; the scoring team must be the defense |


## Setup

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import logging
logging.basicConfig(level=logging.INFO, format='%(message)s')

import pandas as pd
pd.set_option('display.max_columns', 30)
pd.set_option('display.width', 200)

In [2]:
from src.ingest import (
    get_weekly_stats, get_pbp, get_id_crosswalk, get_sleeper_league,
)
from src.features import (
    compute_custom_score, scoring_coverage_report,
    validate_against_sleeper, add_pick_six_column, kicker_miss_audit,
)

## Configuration

`SEASONS` is the history we model on -- 2018-2025 (8 seasons), the default since the Phase 6 data-volume experiment (see `PROJECT_CONTEXT.md`'s Phase 6 findings): Model A's gap against simple baselines closed and reversed sign at every position between 2 and 8 seasons of training history, so more history is worth having by default, not just as a one-off test. `LEAGUE_ID_2025` is used for validation because we're checking against completed 2025 weeks -- the 2026 league has a different id and no results yet.

In [3]:
SEASONS        = list(range(2018, 2026))  # 2018-2025, 8 seasons -- see Configuration note above
LEAGUE_ID_2025 = "1250182471429931008"
LEAGUE_ID_2026 = "1389706592789733376"

VALIDATION_WEEKS = [5, 8, 10, 12, 15]
SKILL_POSITIONS  = ['QB', 'RB', 'WR', 'TE']

## 1. Load data

All cache hits — nothing re-downloads.

In [4]:
weekly    = get_weekly_stats(SEASONS)
pbp       = get_pbp(SEASONS)
crosswalk = get_id_crosswalk()
league    = get_sleeper_league(LEAGUE_ID_2025)
scoring   = league['scoring_settings']

print(f"weekly: {len(weekly):,} rows | pbp: {len(pbp):,} rows")

[cache hit]  weekly_2018_2019_2020_2021_2022_2023_2024_2025.parquet


[cache hit]  pbp_2018_2019_2020_2021_2022_2023_2024_2025.parquet


[cache hit]  id_crosswalk.parquet


[cache hit]  sleeper_league_1250182471429931008.json


weekly: 147,226 rows | pbp: 389,358 rows


## 2. Filter to regular season

Weeks 19+ are playoffs. They're real games with real stats, but usage patterns
differ and fantasy seasons end well before them — training on them would teach
the model from games that don't count.

In [5]:
reg = weekly[weekly['season_type'] == 'REG'].copy()
print(f"{len(reg):,} regular-season rows "
      f"({len(weekly) - len(reg):,} postseason dropped)")

140,750 regular-season rows (6,476 postseason dropped)


## 3. Add pick-sixes from play-by-play

The one active scoring rule with no weekly-stats column. A pick-six is an
interception returned for a touchdown, charged to the passer — and crucially,
**the scoring team must be the defense**. Without that condition, a defender
fumbling the return into his own end zone counts as a pick-six when it's the
opposite.

In [6]:
reg = add_pick_six_column(reg, pbp)
print(f"Pick-sixes across {SEASONS}: {int(reg['pass_int_tds'].sum())}")

Pick-sixes across [2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025]: 291


## 4. Coverage report

Which of the league's active rules can we actually compute? Read this — it's the
difference between a scorer that works and a scorer that works for the rules we
remembered to implement.

DST rules showing as `unmapped` is expected and intentional: team defense needs
pbp aggregation, and it's out of scope for the projection model.

In [7]:
cov = scoring_coverage_report(reg, scoring)
print(cov['status'].value_counts().to_string(), "\n")
cov[cov.status != 'unmapped']

status
unmapped              29
computed              21
computed (derived)     2
known-uncomputable     1 



,rule,weight,status,columns
8,fgm,3.000000,computed,fg_made
9,fgm_yds_over_30,0.100000,computed (derived),fg_made_list
10,fgmiss,-1.000000,computed (derived),fg_missed_list
12,fum,-1.000000,computed,fumbles_total
13,fum_lost,-1.000000,computed,fumbles_lost_total
14,fum_rec,1.000000,known-uncomputable,
15,fum_rec_td,6.000000,computed,fumble_recovery_tds
17,kr_yd,0.030303,computed,kickoff_return_yards
18,pass_2pt,2.000000,computed,passing_2pt_conversions
19,pass_int,-2.000000,computed,passing_interceptions


## 5. Score every player-week

In [8]:
reg['custom_points'] = compute_custom_score(reg, scoring)

reg[reg.position.isin(SKILL_POSITIONS)][
    ['custom_points', 'fantasy_points', 'fantasy_points_ppr']
].describe().round(2)

Non-zero scoring rules with no column mapping: ['fgmiss_50p']. These are silently excluded from the score.


,custom_points,fantasy_points,fantasy_points_ppr
count,45693.00,45693.00,45693.00
mean,7.05,5.95,7.97
std,7.41,7.03,8.10
min,-7.74,-6.66,-6.66
25%,1.35,0.70,1.60
50%,4.50,3.20,5.50
75%,10.76,9.10,12.40
max,54.70,53.20,57.90


## 6. Validate against Sleeper's actual results

This is ground truth, not an approximation — Sleeper ran the league, so its
per-player points for a completed week are definitive. Every row where `diff`
is 0 confirms a rule; every non-zero row points at exactly one rule that's
wrong or missing.

In [9]:
for wk in VALIDATION_WEEKS:
    r = validate_against_sleeper(reg, crosswalk, scoring,
                                 LEAGUE_ID_2025, 2025, wk)
    s = r[r.position.isin(SKILL_POSITIONS)]
    print(f"Wk {wk:>2}: skill {(s['diff'].abs() <= .01).mean():>6.1%} ({len(s):>3})"
          f"  |  all {(r['diff'].abs() <= .01).mean():>6.1%} ({len(r):>3})")

Wk  5: skill 100.0% (131)  |  all 100.0% (146)


Wk  8: skill 100.0% (123)  |  all 100.0% (137)


Wk 10: skill 100.0% (133)  |  all 100.0% (146)


Wk 12: skill 100.0% (132)  |  all 100.0% (146)


Wk 15: skill 100.0% (150)  |  all 100.0% (164)


### Any remaining mismatches

Should be empty. If not, each row names a specific rule to investigate — that's
how every rule in the table at the top was found.

In [10]:
bad = []
for wk in VALIDATION_WEEKS:
    r = validate_against_sleeper(reg, crosswalk, scoring,
                                 LEAGUE_ID_2025, 2025, wk)
    b = r[r['diff'].abs() > 0.01].copy()
    b['week'] = wk
    bad.append(b)

bad = pd.concat(bad, ignore_index=True)
print(f"{len(bad)} mismatched player-weeks")
bad[['week', 'player_display_name', 'position',
     'sleeper_points', 'custom_points', 'diff']]

0 mismatched player-weeks


,week,player_display_name,position,sleeper_points,custom_points,diff


## 7. Save the scored table

Phase 2b (usage and efficiency features) picks up from here.

In [11]:
out = PROJECT_ROOT / 'data' / 'processed' / 'weekly_scored.parquet'
reg.to_parquet(out, index=False)
print(f"Wrote {len(reg):,} rows -> {out.relative_to(PROJECT_ROOT)}")

Wrote 140,750 rows -> data\processed\weekly_scored.parquet


## What's next — Phase 2b

Usage and efficiency features aggregated from play-by-play:

- **Target share, air-yards share** — how much of the offense flows through a player
- **Snap share** — needs the `pfr_player_id` → `gsis_id` crosswalk hop, which
  hasn't been exercised yet. Most likely place for the next surprise.
- **Red-zone touches** — where the touchdowns actually come from
- **aDOT, YAC** — separating volume from efficiency

**Out of scope, deliberately:** K and DST projections. Kicker output depends on
how often the offense stalls in FG range, which is close to noise week to week;
DST would need a team-defense model layered on an offense model. The dashboard
keeps showing Sleeper's numbers for both, labeled as Sleeper's.
